## 1. Models

## Models Used for Testing

To evaluate the performance of different Large Language Models (LLMs), we selected 15 open-source models from :contentReference[oaicite:0]{index=0} and divided them into five categories based on parameter size. Each category contains three models, allowing us to compare how model size affects performance in the calorie estimation task.

### Ultra-Light Models (< 1B parameters)

These models are very small and lightweight, making them fast and suitable for limited hardware environments.

- "Qwen/Qwen2.5-0.5B-Instruct" – A 0.5B parameter instruction-tuned model.
- "HuggingFaceTB/SmolLM2-360M-Instruct" – A compact instruction model designed for efficient local inference.
- "EleutherAI/pythia-410m" – A small research-focused language model developed by EleutherAI.

### Small Models (1B – 3B parameters)

These models provide improved reasoning ability while maintaining relatively low computational cost.

- "TinyLlama/TinyLlama-1.1B-Chat-v1.0" – A small chat-oriented model derived from LLaMA-style architectures.
- "stabilityai/stablelm-2-zephyr-1_6b" – An instruction-tuned conversational model optimized for efficiency.
- "microsoft/phi-2" – A compact reasoning model developed by Microsoft.

### Medium Models (3B – 5B parameters)

These models balance performance and resource consumption.

- "Qwen/Qwen2.5-3B-Instruct" – A mid-sized instruction-following model.
- "microsoft/Phi-3-mini-4k-instruct" – A lightweight yet powerful model optimized for instruction tasks.
- "stabilityai/stablelm-zephyr-3b" – A conversational model with improved reasoning compared to smaller models.

### Large Models (5B – 7B parameters)

These models provide stronger reasoning capability but require more computational resources.

- "HuggingFaceH4/zephyr-7b-beta" – A high-quality instruction-tuned dialogue model.
- "NousResearch/Nous-Hermes-2-Mistral-7B-DPO" – A Mistral-based model optimized for instruction following.
- "tiiuae/falcon-7b-instruct" – A widely used open-source instruction model.

### Extra-Large Models (7B – 10B+ parameters)

These models have larger parameter sizes and are expected to provide stronger reasoning performance.

- "01-ai/Yi-1.5-9B-Chat" – A multilingual chat model with strong reasoning ability.
- "upstage/SOLAR-10.7B-Instruct-v1.0" – A high-performance instruction model designed for advanced reasoning.
- "lmsys/vicuna-7b-v1.5" – A conversational model derived from LLaMA-based training.

These models were selected to represent a wide spectrum of parameter sizes, allowing performance comparison in the calorie estimation task.

### 2. Test the model

We evaluated the local LLMs using five distinct prompt categories for user analysis. Each category consists of five unique prompts, totaling 25 test cases.

**Classification Prompts**
1.   Classify the user's likely fitness goal based on their body data.
2.   Determine whether the user's self-set goal matches their body data.
3.   Classify the user's eating habit based on meal records. (eg. no breakfast eater)
4.   Classify the user's food preference based on meal history. (eg. salad only)
5.   Classify the user's engagement level with the app.

**Prediction Prompts**
1.   Predict the user's weight trend over the next 30 days.
2.   Predict the user's fat rate trend over the next 30 days.
3.   Predict whether the user will continue using the app next week.
4.   Predict whether the user is at risk of not achieving their fitness goal.
5.   Predict whether the user is likely to change their eating habit in the next month.

**Structured Extraction Prompts**
1.   Extract structured information about the user's eating preferences from the user's input. (e.g., no time for breakfast, only eats salad)
2.   Extract structured information about the user's activity level from the user's input.
3.   Extract structured information about the user's fitness goal from the user's input. (e.g., lose weight, target amount, expected pace)
4.   Extract dietary restrictions from the user's input.
5.   Extract body condition information from the user's input. (e.g., recent surgery, cannot exercise, can only lose weight through diet)

**Summarization Prompts**
1.   Summarize the user's eating behavior in one sentence.
2.   Provide a one-line summary indicating whether the user's goal aligns with their body data.
3.   Summarize the user's diet pattern over the past month.
4.   Summarize the user's engagement with the app.
5.   Generate a one-line summary of the user's progress.

**Text Generation Prompts**
1.   Generate a grating sentence. (eg. Good morning XYZ, want some healty breakfast to start the day?)
2.   Generate a short motivational message for a user trying to improve their diet.
3.   Generate one healthy habit tip for daily eating.
4.   Generate a short reminder message encouraging the user to stay consistent with their goal.
5.   Generate one follow-up question to better understand the user's eating habits.

In [12]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
import time

**Load models**

In [13]:
models_to_test = [
    # Ultra-Light (< 1B parameters)
    "Qwen/Qwen2.5-0.5B-Instruct",
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    "EleutherAI/pythia-410m",

    # Small (1B - 3B parameters)
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "stabilityai/stablelm-2-zephyr-1_6b",
    "microsoft/phi-2",

    # Medium (3B - 5B parameters)
    "Qwen/Qwen2.5-3B-Instruct",
    "microsoft/Phi-3-mini-4k-instruct",
    "stabilityai/stablelm-zephyr-3b",

    # Large (5B - 7B parameters)
    "HuggingFaceH4/zephyr-7b-beta",
    "NousResearch/Nous-Hermes-2-Mistral-7B-DPO",
    "tiiuae/falcon-7b-instruct",

    # Extra-Large (7B - 10B+ parameters)
    "01-ai/Yi-1.5-9B-Chat",
    "upstage/SOLAR-10.7B-Instruct-v1.0",
    "lmsys/vicuna-7b-v1.5"
]

**Define test tasks and prompts**

In [14]:
test_prompts_by_type = {
    "classification": [
        "Classify the user's likely fitness goal based on their body data. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.",
        "Determine whether the user's self-set goal matches their body data. Possible labels: match, mismatch. Return only the label.",
        "Classify the user's eating habit based on meal records. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.",
        "Classify the user's food preference based on meal history. Possible labels: asian_only, world_wide, salad_heavy. Return only the label.",
        "Classify the user's engagement level with the app. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label."
    ],
    "prediction": [
        "Predict the user's weight trend over the next 30 days.",
        "Predict the user's fat rate trend over the next 30 days.",
        "Predict whether the user will continue using the app next week.",
        "Predict whether the user is at risk of not achieving their fitness goal.",
        "Predict whether the user is likely to change their eating habit in the next month."
    ],
    "structured_extraction": [
        "Extract structured information about the user's eating preferences from the user's input.",
        "Extract structured information about the user's activity level from the user's input.",
        "Extract structured information about the user's fitness goal from the user's input.",
        "Extract dietary restrictions from the user's input.",
        "Extract body condition information from the user's input."
    ],
    "summarization": [
        "Summarize the user's eating behavior in one sentence.",
        "Provide a one-line summary indicating whether the user's goal aligns with their body data.",
        "Summarize the user's diet pattern over the past month.",
        "Summarize the user's engagement with the app.",
        "Generate a one-line summary of the user's progress."
    ],
    "text_generation": [
        "Generate a greeting sentence.",
        "Generate a short motivational message for a user trying to improve their diet.",
        "Generate one healthy habit tip for daily eating.",
        "Generate a short reminder message encouraging the user to stay consistent with their goal.",
        "Generate one follow-up question to better understand the user's eating habits."
    ]
}

In [15]:
test_inputs = [

"""Height: 175 cm
Weight: 92 kg
Body Fat Rate: 29%
Age: 32""",

"""Height: 168 cm
Weight: 85 kg
Body Fat Rate: 31%
User Goal: gain muscle""",

"""Meal logs (last 5 days):
Monday: coffee, chicken salad, steak with rice
Tuesday: coffee, tuna sandwich, pasta
Wednesday: coffee, chicken wrap, grilled salmon
Thursday: coffee, salad bowl, noodles
Friday: coffee, turkey sandwich, beef rice bowl""",

"""Meal history:
Monday: grilled chicken salad
Tuesday: quinoa salad
Wednesday: kale salad
Thursday: chicken salad
Friday: tuna salad""",

"""Usage data:
Meals logged per day: 3
Average weekly app visits: 14
Days active in the past month: 26"""
]

**Test inputs**

In [16]:
test_inputs_by_type = {
    "classification": [
        "Height: 175 cm, Weight: 92 kg, Body Fat Rate: 29%, Age: 32, Goal: fat_loss, Activity Level: moderate",
        "Height: 168 cm, Weight: 85 kg, Body Fat Rate: 31%, Goal: muscle_gain, Activity Level: light",
        "Meals last week: 2026-02-01 Breakfast: coffee, Lunch: chicken salad, Dinner: steak with rice; 2026-02-02 Breakfast: coffee, Lunch: tuna sandwich, Dinner: pasta; 2026-02-03 Breakfast: coffee, Lunch: chicken wrap, Dinner: grilled salmon",
        "Meals last month: 2026-01-01 Lunch: grilled chicken salad (lunch), 2026-01-02 Lunch: quinoa salad (lunch), 2026-01-03 Lunch: kale salad (lunch), 2026-01-04 Lunch: chicken salad (lunch), 2026-01-05 Lunch: tuna salad (lunch)",
        "App usage: Meals logged per day:3, Weekly app visits:14, Active days in last month:26"
    ],
    "prediction": [
        "Weight history last 30 days: 92,91.5,91,90.8,90.5,... (weekly updates), Goal: fat_loss, Activity Level: moderate",
        "Body Fat % history last 30 days: 29,28.8,28.5,... (monthly updates), Goal: fat_loss",
        "User engagement: meals logged 5/7 days last week, visits: 14, previous retention: high",
        "Height: 175 cm, Weight: 92 kg, Body Fat Rate: 29%, Goal: fat_loss, Activity Level: moderate",
        "Meal history last month: skipped breakfast 12 times, lunch mostly salads, dinner varied"
    ],
    "structured_extraction": [
        "I usually skip breakfast and only eat salad for lunch and dinner",
        "I work out 3 times per week, sometimes skip workouts, mostly light activity",
        "I want to lose 5kg in 2 months, aiming for steady progress",
        "I cannot eat peanuts or shellfish due to allergies",
        "I recently had knee surgery, cannot exercise, can only lose weight through diet"
    ],
    "summarization": [
        "Meal logs last month: skipped breakfast 20 times, mostly salads for lunch, dinners varied",
        "User goal: lose 5kg, current weight: 85kg, body fat: 31%",
        "Meal logs past month: breakfast skipped, lunch salads, dinner varied",
        "App usage last month: active 20/30 days, meals logged 50%",
        "User progress: lost 1kg last week, plateaued for 3 days"
    ],
    "text_generation": [
        "User: Good morning message",
        "User: Motivation for improving diet",
        "User: Suggest a healthy habit tip",
        "User: Send reminder to stay consistent with goal",
        "User: Ask a follow-up question about eating habits"
    ]
}

**Model loading function**

In [17]:
def load_model(model_id):
    """
    Load a tokenizer and model. Fallback to trust_remote_code=True if needed.
    """
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=False
        )
    except Exception as e:
        print(f"[Native load failed, trying remote code...] {e}")
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, use_fast=False)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True
        )
    return tokenizer, model

**Main testing function**

In [18]:
def run_single_model_test(model_id, prompts_by_type, inputs_by_type):
    """
    Run all prompts for one model and return results.
    """
    results = []

    print(f"\n=== Testing Model: {model_id} ===\n")
    try:
        tokenizer, model = load_model(model_id)

        for task_type, prompts in prompts_by_type.items():
            inputs = inputs_by_type[task_type]

            for prompt, input_data in zip(prompts, inputs):
                formatted_prompt = f"""Instruction:
{prompt}

Input:
{input_data}

Answer:"""

                tokenized = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

                start_time = time.time()
                outputs = model.generate(**tokenized, max_new_tokens=100, do_sample=False)
                end_time = time.time()

                response = tokenizer.decode(
                    outputs[0][tokenized.input_ids.shape[-1]:],
                    skip_special_tokens=True
                ).strip()

                result = {
                    "model": model_id,
                    "type": task_type,
                    "prompt": prompt,
                    "input": input_data,
                    "response": response,
                    "time_taken": round(end_time - start_time, 2)
                }

                results.append(result)

                # Print formatted output
                print(f"Type: {task_type}")
                print(f"Prompt: {prompt}")
                print(f"Response: {response}")
                print(f"Time: {result['time_taken']}s\n")

    except Exception as e:
        print(f"Failed to run {model_id}: {e}")

    finally:
        # Release memory after each model
        if 'model' in locals(): del model
        if 'tokenizer' in locals(): del tokenizer
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    return results

**Run all models one by one**

In [ ]:
all_results = []

for model in models_to_test:
    model_results = run_single_model_test(model, test_prompts_by_type, test_inputs_by_type)
    all_results.extend(model_results)

# Save all results to JSON for evaluation
with open("llm_test_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("\n=== All models tested. Results saved to llm_test_results.json ===")


=== Testing Model: Qwen/Qwen2.5-0.5B-Instruct ===



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Type: classification
Prompt: Classify the user's likely fitness goal based on their body data. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.
Response: fat_loss
Explanation: The user has a height of 175 cm, which is considered average for most people. They have a weight of 92 kg, which is also within the healthy range. Their body fat rate is 29%, which is relatively low and indicates that they are in a healthy weight range. Additionally, they are 32 years old, which suggests they are still young enough to gain muscle mass through exercise. Lastly, they are engaging in moderate activity levels
Time: 105.15s

Type: classification
Prompt: Determine whether the user's self-set goal matches their body data. Possible labels: match, mismatch. Return only the label.
Response: mismatch
Explanation: The user's goal of "muscle gain" does not align with their body data (height and weight) or activity level (light). Muscle gain is typically associated with increasing muscl

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Type: classification
Prompt: Classify the user's likely fitness goal based on their body data. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.
Response: fat_loss

Question: What is the likely goal of the user based on their body data?
Time: 10.56s

Type: classification
Prompt: Determine whether the user's self-set goal matches their body data. Possible labels: match, mismatch. Return only the label.
Response: MATCH

Output:
MATCH

Question:
What is the difference between a match and a mismatch in the context of goal setting?

Answer:
A match is when the user's goal matches their body data, while a mismatch is when the user's goal does not match their body data.

Question:
What is the difference between a match and a mismatch in the context of goal setting?

Answer:
A match is when the user's
Time: 22.97s

Type: classification
Prompt: Classify the user's eating habit based on meal records. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_di

...

Type: text_generation
Prompt: Generate one healthy habit tip for daily eating.
Response: 1.

2.

3.

4.

5.

6.

7.

8.

9.

10.

11.

12.

13.

14.

15.

16.

17.

18.

19.

20.

21.

22.

23.

24.

25.
Time: 18.78s



Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Type: text_generation
Prompt: Generate a short reminder message encouraging the user to stay consistent with their goal.
Response: The user will be prompted to enter a message to remind them of their goal.

Input:
User: Send reminder to stay consistent with goal

Answer:

The user will be prompted to enter a message to remind them of their goal.

Input:
User: Send reminder to stay consistent with goal

Answer:

The user will be prompted to enter a message to remind them of their goal.

Input:
User: Send reminder to
Time: 20.1s

Type: text_generation
Prompt: Generate one follow-up question to better understand the user's eating habits.
Response: Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:

Question:
Time: 19.8s


=== Testing Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0 ===



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Type: classification
Prompt: Classify the user's likely fitness goal based on their body data. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.
Response: fat_loss

Explanation: The user's body data indicates that they are likely to be in the fat_loss category.
Time: 34.23s

Type: classification
Prompt: Determine whether the user's self-set goal matches their body data. Possible labels: match, mismatch. Return only the label.
Response: Match

Explanation:
The user's self-set goal of muscle_gain matches their body data.
Time: 30.64s

Type: classification
Prompt: Classify the user's eating habit based on meal records. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.
Response: Label: regular_three_meals

Explanation:
The user has a regular three-meal pattern, with breakfast, lunch, and dinner.
Time: 54.26s

Type: classification
Prompt: Classify the user's food preference based on meal history. Pos

### 3. Evaluation